# EDA del histórico de convalidaciones

Parte A de la Issue #1: carga local de los tres XLS oficiales, deduplicación entre cursos, recuentos, problemas de calidad y selección de las seis universidades con más pares. No se investigan catálogos ni se crea `docs/candidatas.md`.

## 1. Carga y normalización

Se utiliza `pandas.read_excel` con `engine="xlrd"`. La hoja informativa vacía se omite. Los encabezados antiguos presentan caracteres mal decodificados (`C�D.`, `PA�S`), por lo que se renombran por posición, estable en los tres ficheros. La normalización elimina tildes, convierte a mayúsculas y compacta espacios antes de agrupar.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
columns = ["country", "university", "degree", "uc_code", "uc_subject", "ects", "course", "subject_type", "dest_code", "dest_subject"]
frames = []
input_files = sorted(RAW_DIR.glob("*.xls"))
assert len(input_files) == 3, f"Se esperaban 3 XLS y se encontraron {len(input_files)}"
for path in input_files:
    for sheet in pd.ExcelFile(path, engine="xlrd").sheet_names:
        frame = pd.read_excel(path, sheet_name=sheet, engine="xlrd", dtype=object)
        if frame.empty:
            continue
        frame.columns = columns
        frame["source_file"] = path.stem
        frame["academic_year"] = sheet
        frames.append(frame)
historical = pd.concat(frames, ignore_index=True)
print(f"Ficheros cargados: {len(input_files)}; filas: {len(historical):,}")
display(historical.groupby("source_file").size().rename("rows").to_frame())
display(historical.groupby("academic_year").size().rename("rows").to_frame())

def normalize_text(value):
    if pd.isna(value):
        return ""
    text = str(value).replace("\xa0", " ")
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", text).strip().upper()

for column in ["country", "university", "degree", "uc_code", "uc_subject", "dest_code", "dest_subject"]:
    historical[f"{column}_norm"] = historical[column].map(normalize_text)
historical["uc_identity"] = historical["uc_code_norm"].where(historical["uc_code_norm"].ne(""), "NAME:" + historical["uc_subject_norm"])
historical["dest_identity"] = historical["dest_code_norm"].where(historical["dest_code_norm"].ne(""), "NAME:" + historical["dest_subject_norm"])
dedup_key = ["source_file", "country_norm", "university_norm", "degree_norm", "uc_identity", "dest_identity"]
exact_key = dedup_key + ["academic_year"]
cross_year_group = historical.groupby(dedup_key)["academic_year"].transform("nunique") > 1
repeated_row = historical.duplicated(dedup_key, keep=False)
historical_unique = historical.drop_duplicates(dedup_key).copy()
print(f"Filas implicadas en repeticiones entre cursos: {cross_year_group.sum():,}")
print(f"Filas implicadas en cualquier repetición del mismo par: {repeated_row.sum():,}")
print(f"Filas eliminadas al deduplicar: {len(historical) - len(historical_unique):,}")
print(f"Duplicados exactos dentro del mismo curso: {historical.duplicated(exact_key).sum():,}")

Ficheros cargados: 3; filas: 14,522


,rows
source_file,
Asignaturas_CCSSJJ,6943
Asignaturas_EPS,6780
Asignaturas_HCD,799


,rows
academic_year,
2024-2025,4991
2025-2026,5198
2026-2027,4333


Filas implicadas en repeticiones entre cursos: 3,896
Filas implicadas en cualquier repetición del mismo par: 5,262
Filas eliminadas al deduplicar: 3,157
Duplicados exactos dentro del mismo curso: 1,439


## 2. ¿Dónde hay más datos?

Los recuentos se calculan sobre `historical_unique`, de modo que un mismo par repetido en varios cursos cuenta una sola vez. Un par se identifica por universidad, grado, código UC3M y código destino; si falta uno de los códigos, se utiliza el nombre normalizado de esa asignatura como respaldo. Así, un cambio de nombre conservando los códigos no crea un par nuevo. La tabla agregada se guarda en `data/processed`.

In [2]:
degree_university = (historical_unique.groupby(["degree_norm", "university_norm"], as_index=False).size().rename(columns={"size": "n_convalidaciones"}).sort_values("n_convalidaciones", ascending=False))
degree_university.to_csv(PROCESSED_DIR / "historical_by_degree_university.csv", index=False)
top_universities = historical_unique.groupby("university_norm").size().sort_values(ascending=False).head(10).rename("n_convalidaciones").to_frame()
top_degrees = historical_unique.groupby("degree_norm").size().sort_values(ascending=False).head(10).rename("n_convalidaciones").to_frame()
print("Top 10 universidades"); display(top_universities)
print("Top 10 grados"); display(top_degrees)
print("Top 10 pares grado × universidad"); display(degree_university.head(10))
print(f"CSV guardado en: {PROCESSED_DIR / 'historical_by_degree_university.csv'}")

Top 10 universidades


,n_convalidaciones
university_norm,
UNIVERSITY OF CALIFORNIA,1498
BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,460
PURDUE UNIVERSITY,361
UNIVERSIDAD TORCUATO DI TELLA,280
GEORGIA INSTITUTE OF TECHNOLOGY,276
UNIVERSITY OF MARYLAND,272
UNIVERSITY OF WATERLOO,226
CONCORDIA UNIVERSITY,225
GEORGE MASON UNIVERSITY,216


Top 10 grados


,n_convalidaciones
degree_norm,
DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMPRESAS,1163
GRADO EN INGENIERIA EN TECNOLOGIAS INDUSTRIALES,893
DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y DERECHO,559
DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y ECONOMIA,533
DOBLE GRADO EN DERECHO Y ECONOMIA,524
GRADO EN EMPRESA Y TECNOLOGIA,520
GRADO EN INGENIERIA INFORMATICA,501
DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y ADMINISTRACION DE EMPRESAS,472
GRADO EN INGENIERIA BIOMEDICA,472


Top 10 pares grado × universidad


,degree_norm,university_norm,n_convalidaciones
748,GRADO EN INGENIERIA EN TECNOLOGIAS INDUSTRIALES,UNIVERSITY OF CALIFORNIA,142
25,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,128
234,DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y DERECHO,UNIVERSITY OF CALIFORNIA,119
67,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,UNIVERSITY OF CALIFORNIA,118
265,DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y ECON...,UNIVERSITY OF CALIFORNIA,115
246,DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y ECON...,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,106
286,DOBLE GRADO EN INGENIERIA FISICA E INGENIERIA ...,UNIVERSITY OF CALIFORNIA,88
142,DOBLE GRADO EN DERECHO Y ECONOMIA,UNIVERSITY OF CALIFORNIA,79
315,DOBLE GRADO EN INGENIERIA INFORMATICA Y ADMINI...,UNIVERSITY OF CALIFORNIA,75
63,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,UNIVERSIDAD TORCUATO DI TELLA,74


CSV guardado en: /Users/kells/aisc-convalidaciones/data/processed/historical_by_degree_university.csv


### Limitación: unidad de contabilización

Los recuentos se hacen por **titulación**: el grado normalizado forma parte de la clave junto con la universidad y la pareja de asignaturas. Por ello, una misma universidad puede aparecer varias veces si ofrece convalidaciones para titulaciones distintas; no se está midiendo únicamente el volumen por institución agregada.

## 3. Problemas de los datos y ejemplos

Se cuantifican filas repetidas, relaciones n:m, filas sin código destino y nombres de universidad inconsistentes. Los ejemplos se generan directamente de las filas afectadas. Las relaciones n:m se calculan solo con códigos destino no vacíos.

In [3]:
print("### Filas repetidas entre cursos")
repeat_example = historical.loc[cross_year_group, ["academic_year", "university", "degree", "uc_code", "dest_code", "dest_subject"]].sort_values(["university", "degree", "uc_code", "academic_year"]).head(10)
print(f"Filas implicadas en más de un curso: {cross_year_group.sum():,}"); display(repeat_example)

with_codes = historical_unique[(historical_unique.uc_code_norm != "") & (historical_unique.dest_code_norm != "")]
uc_to_dest = with_codes.groupby(["source_file", "university_norm", "degree_norm", "uc_identity"]).dest_identity.agg(lambda values: sorted(set(values))).reset_index(name="dest_codes")
uc_to_dest["n_dest"] = uc_to_dest.dest_codes.map(len)
dest_to_uc = with_codes.groupby(["source_file", "university_norm", "degree_norm", "dest_identity"]).uc_identity.agg(lambda values: sorted(set(values))).reset_index(name="uc_codes")
dest_to_uc["n_uc"] = dest_to_uc.uc_codes.map(len)
nm_uc_to_dest = uc_to_dest[uc_to_dest.n_dest > 1]
nm_dest_to_uc = dest_to_uc[dest_to_uc.n_uc > 1]
print("### Relaciones n:m")
print(f"Grupos UC3M → varios destinos: {len(nm_uc_to_dest):,}")
print(f"Grupos destino → varias UC3M: {len(nm_dest_to_uc):,}")
display(nm_uc_to_dest.sort_values("n_dest", ascending=False).head(3))
display(nm_dest_to_uc.sort_values("n_uc", ascending=False).head(3))

missing_dest_code = historical_unique[historical_unique.dest_code_norm == ""]
print("### Asignaturas sin código de destino")
print(f"Filas únicas sin código destino: {len(missing_dest_code):,}")
display(missing_dest_code[["university", "degree", "uc_code", "uc_subject", "dest_subject"]].head(10))

university_variants = historical.assign(raw_university=historical.university.fillna("").astype(str).str.strip()).groupby("university_norm").raw_university.agg(lambda values: sorted(set(values)))
university_variants = university_variants[university_variants.map(len) > 1]
print("### Nombres de universidad inconsistentes")
print(f"Universidades con varias escrituras: {len(university_variants):,}")
display(university_variants.rename("raw_variants").to_frame().head(10))

california_rows = historical[historical.university_norm.str.contains("CALIFORNIA", na=False)]
print("### Comprobación de campus de University of California")
display(california_rows[["university", "university_norm"]].drop_duplicates().sort_values("university_norm"))
campus_markers = r"BERKELEY|IRVINE|SAN DIEGO|LOS ANGELES|DAVIS|SANTA BARBARA|RIVERSIDE|MERCED|SANTA CRUZ"
print("Valores con marcadores de campus entre todos los nombres:")
display(historical.loc[historical.university.astype(str).str.contains(campus_markers, case=False, regex=True, na=False), "university"].drop_duplicates().sort_values().to_frame())
replacement_counts = historical.astype(str).apply(lambda column: column.str.contains("�", regex=False).sum())
print("### Observación adicional de codificación")
display(replacement_counts[replacement_counts > 0].rename("n_celdas_con_�").to_frame())

### Filas repetidas entre cursos
Filas implicadas en más de un curso: 3,896


,academic_year,university,degree,uc_code,dest_code,dest_subject
14041,2025-2026,AMERICAN UNIVERSITY OF CAIRO,Grado en Historia y Política,17699,ANTH 2101,Cultural Anthropology
14223,2026-2027,AMERICAN UNIVERSITY OF CAIRO,Grado en Historia y Política,17699,ANTH 2101,Cultural Anthropology
14042,2025-2026,AMERICAN UNIVERSITY OF CAIRO,Grado en Historia y Política,17700,POL 4526,Political Economy of the Global South
14224,2026-2027,AMERICAN UNIVERSITY OF CAIRO,Grado en Historia y Política,17700,POL 4526,Political Economy of the Global South
9780,2025-2026,ARIZONA STATE UNIVERSITY,Grado en Ingeniería Informática,13882,CSE 463,Introduction to Human Computer Interaction
12224,2026-2027,ARIZONA STATE UNIVERSITY,Grado en Ingeniería Informática,13882,CSE 463,Introduction to Human Computer Interaction
7249,2024-2025,ARIZONA STATE UNIVERSITY,Grado en Ingeniería Informática,13884,CSE 434,Computer Networks
9782,2025-2026,ARIZONA STATE UNIVERSITY,Grado en Ingeniería Informática,13884,CSE 434,Computer Networks
12225,2026-2027,ARIZONA STATE UNIVERSITY,Grado en Ingeniería Informática,13884,CSE 434,Computer Networks
7250,2024-2025,ARIZONA STATE UNIVERSITY,Grado en Ingeniería Informática,13888,CSE 420,Computer Architecture I


### Relaciones n:m
Grupos UC3M → varios destinos: 1,896
Grupos destino → varias UC3M: 834


,source_file,university_norm,degree_norm,uc_identity,dest_codes,n_dest
2611,Asignaturas_CCSSJJ,UNIVERSITY OF CALIFORNIA,DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y DERECHO,16635,"[102, GLBL ST 160, GLOBAL 110Q 001, GLOBAL 178...",13
2613,Asignaturas_CCSSJJ,UNIVERSITY OF CALIFORNIA,DOBLE GRADO EN ESTUDIOS INTERNACIONALES Y DERECHO,16637,"[100H, DEMOG 110 001, DEMOG C126 001, DEMOG C1...",12
123,Asignaturas_CCSSJJ,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,13473,"[ACC 3000, ECO 3220, FIN 3000, LAW 3102, LAW 3...",12


,source_file,university_norm,degree_norm,dest_identity,uc_codes,n_uc
155,Asignaturas_CCSSJJ,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,TAX 3300,"[13164, 13167, 13170, 13174, 13180, 13473, 135...",11
154,Asignaturas_CCSSJJ,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,RES 3000,"[13164, 13167, 13170, 13174, 13180, 13473, 135...",10
145,Asignaturas_CCSSJJ,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,DOBLE GRADO EN DERECHO Y ADMINISTRACION DE EMP...,MKT 5750,"[13164, 13167, 13170, 13174, 13180, 13473, 135...",10


...

,university,university_norm
7258,CALIFORNIA STATE UNIVERSITY - LONG BEACH,CALIFORNIA STATE UNIVERSITY - LONG BEACH
1492,UNIVERSITY OF CALIFORNIA,UNIVERSITY OF CALIFORNIA
1943,UNIVERSITY OF SOUTHERN CALIFORNIA-MARSHALL BUS...,UNIVERSITY OF SOUTHERN CALIFORNIA-MARSHALL BUS...


Valores con marcadores de campus entre todos los nombres:


,university
1287,SAN DIEGO STATE UNIVERSITY


### Observación adicional de codificación


,n_celdas_con_�


## 4. Seis universidades prioritarias para la parte B

La lista siguiente solo identifica las seis universidades con más pares históricos deduplicados. La investigación de sus catálogos se hará fuera de este entregable.

### Limitaciones de interpretación

Las relaciones **n:m** detectadas son posibles relaciones múltiples derivadas de las filas del histórico: indican que varios códigos aparecen relacionados, pero no confirman por sí solas una agrupación académica formal ni una equivalencia conjunta aprobada.

Asimismo, `UNIVERSITY OF CALIFORNIA` no tiene campus identificado en los datos originales. El análisis conserva literalmente ese nombre y no puede distinguir Berkeley, Irvine, San Diego u otros campus cuando la fuente no los especifica.

In [4]:
top_six_universities = top_universities.head(6)
display(top_six_universities)
for position, university in enumerate(top_six_universities.index, start=1):
    print(f"{position}. {university}")

,n_convalidaciones
university_norm,
UNIVERSITY OF CALIFORNIA,1498
BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,460
PURDUE UNIVERSITY,361
UNIVERSIDAD TORCUATO DI TELLA,280
GEORGIA INSTITUTE OF TECHNOLOGY,276
UNIVERSITY OF MARYLAND,272


1. UNIVERSITY OF CALIFORNIA
2. BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK
3. PURDUE UNIVERSITY
4. UNIVERSIDAD TORCUATO DI TELLA
5. GEORGIA INSTITUTE OF TECHNOLOGY
6. UNIVERSITY OF MARYLAND


## 5. Calidad de los datos de las seis universidades

Para elegir el piloto a parte del número de pares hay que ver también cómo de limpios están. Para cada una de las seis universidades se calcula:

- **pares**: pares deduplicados entre cursos.
- **% repetidos entre cursos**: porcentaje de filas originales cuyo par aparece en más de un curso (indica estabilidad; la deduplicación ya los elimina).
- **% sin código destino**: porcentaje de pares sin código de la asignatura destino.
- **% n:m**: porcentaje de asignaturas UC3M (por grado) relacionadas con más de un código destino. Son relaciones posibles, no equivalencias conjuntas confirmadas.

In [5]:
top6 = top_six_universities.index
rows = []
for uni in top6:
    u = historical_unique[historical_unique.university_norm == uni]
    h = historical[historical.university_norm == uni]
    rows.append({
        "universidad": uni,
        "pares": len(u),
        "% repetidos entre cursos": round(100 * cross_year_group[h.index].mean(), 1),
        "% sin codigo destino": round(100 * (u.dest_code_norm == "").mean(), 1),
        "% n:m (UC3M a varios destinos)": round(
            100 * uc_to_dest[uc_to_dest.university_norm == uni].n_dest.gt(1).mean(), 1),
    })
display(pd.DataFrame(rows))

,universidad,pares,% repetidos entre cursos,% sin codigo destino,% n:m (UC3M a varios destinos)
0,UNIVERSITY OF CALIFORNIA,1498,24.9,0.0,64.0
1,BARUCH COLLEGE-THE CITY UNIVERSITY OF NEW YORK,460,29.6,0.0,52.4
2,PURDUE UNIVERSITY,361,42.7,0.0,29.7
3,UNIVERSIDAD TORCUATO DI TELLA,280,28.9,5.7,38.5
4,GEORGIA INSTITUTE OF TECHNOLOGY,276,39.2,0.0,32.2
5,UNIVERSITY OF MARYLAND,272,50.8,0.0,35.0
